In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..")
df = pd.read_csv(ROOT / "data/staging/stg_harga_harian.csv", parse_dates=["tanggal"])
df = df.sort_values(["komoditas", "pasar", "tanggal"]).reset_index(drop=True)

print(f"{len(df):,} baris | {df['tanggal'].nunique()} tanggal | {df['pasar'].nunique()} pasar | "
      f"{df['komoditas'].nunique()} komoditas | {df['kelompok'].nunique()} kelompok")
print(df["tanggal"].min().date(), "->", df["tanggal"].max().date())
print("Duplikat tanggal+pasar+komoditas:", df.duplicated(["tanggal", "pasar", "komoditas"]).sum())
print("Harga <= 0 atau kosong:", int((df["harga"].isna() | (df["harga"] <= 0)).sum()))

(df.groupby("komoditas")
   .agg(kelompok=("kelompok", "first"), satuan=("satuan", "first"), baris=("harga", "size"),
        min=("harga", "min"), median=("harga", "median"), maks=("harga", "max")).round(0))

91,543 baris | 1332 tanggal | 7 pasar | 10 komoditas | 7 kelompok
2023-01-01 -> 2026-08-31
Duplikat tanggal+pasar+komoditas: 0
Harga <= 0 atau kosong: 0


,kelompok,satuan,baris,min,median,maks
komoditas,,,,,,
Bawang Bombay,BAWANG,Kg,9158,15000.0,30000.0,70000.0
Buncis,SAYUR-SAYURAN,Kg,9076,5000.0,13000.0,30000.0
Jagung Pipilan Kering (Jenis Kecil),JAGUNG,Kg,8908,5000.0,10750.0,15000.0
Kacang Hijau,KACANG-KACANGAN,Kg,9324,17000.0,23000.0,35000.0
Kacang Tanah (Jenis Besar),KACANG-KACANGAN,Kg,9324,19000.0,28000.0,45000.0
Ketela Pohon,UBI-UBIAN,Kg,9002,4000.0,6000.0,8000.0
Telur Ayam Kampung,TELUR,Butir,9323,1800.0,3000.0,3500.0
Tomat,SAYUR-SAYURAN,Kg,9236,3000.0,12000.0,38000.0
Udang Segar (Uk. Sedang),IKAN,Kg,9190,40000.0,65000.0,90000.0


In [2]:
df["bulan"] = df["tanggal"].dt.to_period("M")
hari_bulan = df.groupby("bulan")["tanggal"].nunique()

lengkap = (df.groupby(["bulan", "pasar", "komoditas"])["tanggal"].nunique()
             .groupby(["bulan", "pasar"]).mean()
             .unstack("pasar").div(hari_bulan, axis=0).mul(100).round(0))

print("Bulan terburuk per pasar (%):")
print(lengkap.min().sort_values())

Bulan terburuk per pasar (%):
pasar
Balongsari       74.0
Genteng Baru     84.0
Tambahrejo       93.0
Kembang          99.0
Wonokromo        99.0
Pabean          100.0
Pucang Anom     100.0
dtype: float64


In [3]:
BARU = ["Wortel (Kualitas Lokal)", "Jagung Pipilan Kering (Jenis Kecil)", "Ketela Pohon"]   # yang belum dicek manual

d = df[df["komoditas"].isin(BARU)].copy()
g = d.groupby(["komoditas", "pasar"])["harga"]
d["sebelum"], d["sesudah"] = g.shift(1), g.shift(-1)
d["median_hari"] = d.groupby(["tanggal", "komoditas"])["harga"].transform("median")
d["rasio"] = (d["harga"] / d["median_hari"]).round(2)
d["tab"] = d["tanggal"].dt.day

cek = (d[d["rasio"].lt(0.67) | d["rasio"].gt(1.5)]
         .assign(skor=lambda x: np.abs(np.log(x["rasio"])))
         .sort_values("skor", ascending=False).groupby("komoditas").head(15)
         .sort_values(["komoditas", "tanggal", "pasar"]))

print(cek.groupby("komoditas").size())
cek[["komoditas", "tanggal", "pasar", "harga", "median_hari", "rasio", "sebelum", "sesudah", "sumber_file", "tab"]]

komoditas
Jagung Pipilan Kering (Jenis Kecil)    15
Ketela Pohon                           15
Wortel (Kualitas Lokal)                15
dtype: int64


,komoditas,tanggal,pasar,harga,median_hari,rasio,sebelum,sesudah,sumber_file,tab
20963,Jagung Pipilan Kering (Jenis Kecil),2023-07-01,Kembang,5000.0,9000.0,0.56,10000.0,5000.0,07. REKAP JULI 2023.xlsx,1
20964,Jagung Pipilan Kering (Jenis Kecil),2023-07-02,Kembang,5000.0,9000.0,0.56,5000.0,10000.0,07. REKAP JULI 2023.xlsx,2
23845,Jagung Pipilan Kering (Jenis Kecil),2024-02-04,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,4
23846,Jagung Pipilan Kering (Jenis Kecil),2024-02-05,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,5
23847,Jagung Pipilan Kering (Jenis Kecil),2024-02-06,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,6
23848,Jagung Pipilan Kering (Jenis Kecil),2024-02-07,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,7
23849,Jagung Pipilan Kering (Jenis Kecil),2024-02-08,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,8
23850,Jagung Pipilan Kering (Jenis Kecil),2024-02-09,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,9
23851,Jagung Pipilan Kering (Jenis Kecil),2024-02-10,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,10
23852,Jagung Pipilan Kering (Jenis Kecil),2024-02-11,Pucang Anom,8000.0,12500.0,0.64,8000.0,8000.0,02. REKAP FEBRUARI 2024.xlsx,11


In [4]:
nas = (df.groupby(["komoditas", "tanggal"])["harga"]
         .agg(rata="mean", n_pasar="nunique").reset_index())

desk = nas.groupby("komoditas")["rata"].agg(
    hari="count", rata_rata="mean", simpangan="std", minimum="min",
    q1=lambda s: s.quantile(.25), median="median",
    q3=lambda s: s.quantile(.75), maksimum="max")
desk["cv_%"] = desk["simpangan"] / desk["rata_rata"] * 100
desk["satuan"] = df.groupby("komoditas")["satuan"].first()
desk.round(1).sort_values("cv_%", ascending=False)

,hari,rata_rata,simpangan,minimum,q1,median,q3,maksimum,cv_%,satuan
komoditas,,,,,,,,,,
Tomat,1332,11965.8,4177.1,5571.4,8857.1,11571.4,13428.6,29142.9,34.9,Kg
Bawang Bombay,1332,31621.7,4500.0,21000.0,29428.6,31857.1,33428.6,43285.7,14.2,Kg
Buncis,1332,13846.5,1969.2,9857.1,12142.9,13571.4,15142.9,22142.9,14.2,Kg
Wortel (Kualitas Lokal),1332,12592.4,1669.0,9500.0,11285.7,12285.7,13571.4,17166.7,13.3,Kg
Jagung Pipilan Kering (Jenis Kecil),1332,11448.3,1406.1,8571.4,9500.0,12000.0,12428.6,13166.7,12.3,Kg
Kacang Hijau,1332,24162.7,2441.0,20428.6,22428.6,23142.9,25571.4,30142.9,10.1,Kg
Kacang Tanah (Jenis Besar),1332,29285.9,2567.6,26428.6,27714.3,28428.6,29142.9,37714.3,8.8,Kg
Telur Ayam Kampung,1332,2777.9,156.6,2314.3,2742.9,2828.6,2885.7,2957.1,5.6,Butir
Ketela Pohon,1332,5796.6,251.6,5142.9,5571.4,5714.3,6000.0,6428.6,4.3,Kg


In [5]:
p = nas.pivot(index="tanggal", columns="komoditas", values="rata").asfreq("D").ffill(limit=3)  # tutup bolong <= 3 hari
terakhir = p.dropna().index.max()                      # tanggal terbaru yang lengkap untuk 10 komoditas

def ubah(hari):                                        # perubahan % terhadap `hari` hari sebelumnya
    return ((p / p.shift(hari) - 1) * 100).loc[terakhir]

kpi = pd.DataFrame({"harga_terkini": p.loc[terakhir],
                    "harian_%": ubah(1), "mingguan_%": ubah(7),
                    "bulanan_%": ubah(30), "tahunan_%": ubah(365)})
kpi["volatilitas_30h_%"] = (p.pct_change().rolling(30).std() * 100).loc[terakhir]   # simpangan perubahan harian

print("Tanggal acuan:", terakhir.date())
kpi.round(1)

Tanggal acuan: 2026-08-31


,harga_terkini,harian_%,mingguan_%,bulanan_%,tahunan_%,volatilitas_30h_%
komoditas,,,,,,
Bawang Bombay,31428.6,-0.8,3.3,1.4,-3.5,2.5
Buncis,17500.0,-0.9,0.0,6.1,44.1,5.5
Jagung Pipilan Kering (Jenis Kecil),13166.7,3.9,0.0,3.9,9.7,2.2
Kacang Hijau,28000.0,-1.0,-3.0,-1.0,19.5,1.1
Kacang Tanah (Jenis Besar),36428.6,0.0,-3.4,6.7,32.1,2.2
Ketela Pohon,6000.0,0.0,5.0,-2.3,2.4,3.0
Telur Ayam Kampung,2928.6,0.0,0.0,0.0,2.5,0.0
Tomat,9666.7,0.0,1.8,-7.9,4.1,5.5
Udang Segar (Uk. Sedang),66714.3,-1.6,1.7,-3.7,6.6,1.9


In [6]:
for k in ["Telur Ayam Kampung", "Tomat"]:
    t = nas.loc[nas["komoditas"].eq(k)].nlargest(1, "rata")["tanggal"].iloc[0]
    w = df[(df["komoditas"] == k) & df["tanggal"].between(t - pd.Timedelta(days=2), t + pd.Timedelta(days=2))]
    print(k, "| hari puncak:", t.date())
    display(w.pivot(index="tanggal", columns="pasar", values="harga"))

Telur Ayam Kampung | hari puncak: 2024-03-21


pasar,Balongsari,Genteng Baru,Kembang,Pabean,Pucang Anom,Tambahrejo,Wonokromo
tanggal,,,,,,,
2024-03-19,3000.0,2800.0,3000.0,3000.0,3200.0,3000.0,2500.0
2024-03-20,3000.0,2800.0,3000.0,3000.0,3200.0,3000.0,2500.0
2024-03-21,3000.0,3000.0,3000.0,3000.0,3200.0,3000.0,2500.0
2024-03-22,3000.0,3000.0,3000.0,3000.0,3200.0,3000.0,2500.0
2024-03-23,3000.0,3000.0,3000.0,3000.0,3200.0,3000.0,2500.0


Tomat | hari puncak: 2024-04-08


pasar,Balongsari,Genteng Baru,Kembang,Pabean,Pucang Anom,Tambahrejo,Wonokromo
tanggal,,,,,,,
2024-04-06,30000.0,28000.0,35000.0,20000.0,35000.0,28000.0,10000.0
2024-04-07,30000.0,28000.0,35000.0,20000.0,35000.0,28000.0,10000.0
2024-04-08,38000.0,28000.0,35000.0,35000.0,30000.0,28000.0,10000.0
2024-04-09,38000.0,28000.0,35000.0,35000.0,30000.0,28000.0,10000.0
2024-04-10,38000.0,28000.0,35000.0,35000.0,30000.0,28000.0,10000.0


In [7]:
d = df.copy()
d["median_hari"] = d.groupby(["tanggal", "komoditas"])["harga"].transform("median")
d["rasio"] = (d["harga"] / d["median_hari"]).round(2)
d["tab"] = d["tanggal"].dt.day

kandidat = d[(d["rasio"] >= 5) | (d["rasio"] <= 0.2)].sort_values(["komoditas", "tanggal", "pasar"])
print("Kandidat salah ketik (>= 5x atau <= 0,2x median hari itu):", len(kandidat))
kandidat[["komoditas", "tanggal", "pasar", "harga", "median_hari", "rasio", "sumber_file", "tab"]]

Kandidat salah ketik (>= 5x atau <= 0,2x median hari itu): 0


,komoditas,tanggal,pasar,harga,median_hari,rasio,sumber_file,tab


In [8]:
# Hari puncak dan terendah per komoditas (harga nasional harian)
ekstrem = nas.groupby("komoditas")["rata"].agg(
    puncak="max", tgl_puncak=lambda s: nas.loc[s.idxmax(), "tanggal"].date(),
    terendah="min", tgl_terendah=lambda s: nas.loc[s.idxmin(), "tanggal"].date())
ekstrem["naik_x"] = (ekstrem["puncak"] / ekstrem["terendah"]).round(1)

# Pola musiman: bulan kalender dengan harga rata-rata tertinggi dan terendah
nas["bln"] = nas["tanggal"].dt.month
musim = nas.groupby(["komoditas", "bln"])["rata"].mean().unstack("bln")
musim_idx = musim.div(musim.mean(axis=1), axis=0).mul(100).round(0)   # 100 = rata-rata tahunan

print(ekstrem.round(0).sort_values("naik_x", ascending=False))
musim_idx.style.background_gradient(cmap="RdYlGn_r", axis=None, vmin=70, vmax=130)

                                      puncak  tgl_puncak  terendah  \
komoditas                                                            
Tomat                                29143.0  2024-04-08    5571.0   
Bawang Bombay                        43286.0  2024-05-19   21000.0   
Buncis                               22143.0  2024-03-23    9857.0   
Jagung Pipilan Kering (Jenis Kecil)  13167.0  2026-06-05    8571.0   
Wortel (Kualitas Lokal)              17167.0  2025-02-12    9500.0   
Kacang Hijau                         30143.0  2026-06-12   20429.0   
Ketela Pohon                          6429.0  2025-10-27    5143.0   
Kacang Tanah (Jenis Besar)           37714.0  2026-08-23   26429.0   
Telur Ayam Kampung                    2957.0  2024-03-21    2314.0   
Udang Segar (Uk. Sedang)             70833.0  2026-07-19   60429.0   

                                    tgl_terendah  naik_x  
komoditas                                                 
Tomat                                 202

bln,1,2,3,4,5,6,7,8,9,10,11,12
komoditas,,,,,,,,,,,,
Bawang Bombay,89.000000,88.000000,92.000000,103.000000,118.000000,116.000000,108.000000,100.000000,98.000000,97.000000,95.000000,95.000000
Buncis,93.000000,90.000000,105.000000,106.000000,106.000000,112.000000,96.000000,92.000000,88.000000,97.000000,110.000000,105.000000
Jagung Pipilan Kering (Jenis Kecil),96.000000,100.000000,101.000000,103.000000,102.000000,103.000000,102.000000,102.000000,97.000000,98.000000,98.000000,100.000000
Kacang Hijau,100.000000,101.000000,104.000000,102.000000,103.000000,103.000000,102.000000,102.000000,95.000000,95.000000,96.000000,97.000000
Kacang Tanah (Jenis Besar),103.000000,101.000000,102.000000,100.000000,99.000000,100.000000,100.000000,103.000000,96.000000,95.000000,97.000000,104.000000
Ketela Pohon,101.000000,101.000000,100.000000,101.000000,101.000000,99.000000,100.000000,99.000000,96.000000,99.000000,101.000000,103.000000
Telur Ayam Kampung,99.000000,98.000000,100.000000,100.000000,99.000000,101.000000,103.000000,102.000000,100.000000,99.000000,99.000000,101.000000
Tomat,106.000000,107.000000,113.000000,138.000000,124.000000,114.000000,105.000000,77.000000,61.000000,62.000000,89.000000,101.000000
Udang Segar (Uk. Sedang),100.000000,100.000000,100.000000,101.000000,101.000000,99.000000,100.000000,101.000000,99.000000,101.000000,100.000000,98.000000


In [9]:
kpi.round(2).to_csv(ROOT / "data/processed/kpi_acuan.csv", encoding="utf-8-sig")
desk.round(2).to_csv(ROOT / "data/processed/statistik_acuan.csv", encoding="utf-8-sig")

In [10]:
nas["tahun"] = nas["tanggal"].dt.year
bln = nas.groupby(["komoditas", "tahun", "bln"])["rata"].mean().reset_index()
bln["indeks"] = bln["rata"] / bln.groupby(["komoditas", "tahun"])["rata"].transform("mean") * 100

for k in ["Tomat", "Bawang Bombay", "Buncis", "Wortel (Kualitas Lokal)"]:
    t = bln[bln["komoditas"] == k].pivot(index="tahun", columns="bln", values="indeks")
    print(k)
    display(t.style.format("{:.0f}").background_gradient(cmap="RdYlGn_r", axis=None, vmin=60, vmax=140))

Tomat


bln,1,2,3,4,5,6,7,8,9,10,11,12
tahun,,,,,,,,,,,,
2023,110,104,115,145,137,123,84,81,76,69,65,90
2024,109,125,140,184,142,93,57,46,45,59,96,102
2025,105,101,93,94,99,127,176,95,61,55,94,101
2026,94,88,94,113,110,111,101,88,nan,nan,nan,nan


Bawang Bombay


bln,1,2,3,4,5,6,7,8,9,10,11,12
tahun,,,,,,,,,,,,
2023,75,74,81,118,142,133,106,99,95,94,93,91
2024,82,85,90,95,124,123,112,101,101,97,93,96
2025,93,91,91,98,106,108,114,103,99,101,98,99
2026,102,101,102,102,102,99,97,96,nan,nan,nan,nan


Buncis


bln,1,2,3,4,5,6,7,8,9,10,11,12
tahun,,,,,,,,,,,,
2023,88,90,97,105,117,112,99,81,90,94,115,115
2024,94,85,126,118,109,103,86,83,87,97,106,106
2025,104,96,93,91,94,125,94,88,91,106,117,102
2026,86,90,100,108,102,105,101,110,nan,nan,nan,nan


Wortel (Kualitas Lokal)


bln,1,2,3,4,5,6,7,8,9,10,11,12
tahun,,,,,,,,,,,,
2023,100,93,93,96,96,96,98,100,107,109,105,106
2024,106,110,108,109,109,106,107,92,87,87,90,89
2025,107,119,106,101,90,89,88,85,85,98,109,123
2026,108,102,104,95,83,99,105,106,nan,nan,nan,nan


In [11]:
sel = df.merge(nas[["komoditas", "tanggal", "rata"]], on=["komoditas", "tanggal"])
sel["selisih_%"] = (sel["harga"] / sel["rata"] - 1) * 100

tab = sel.groupby(["komoditas", "pasar"])["selisih_%"].mean().unstack("pasar").round(1)
tab.style.background_gradient(cmap="RdYlGn_r", axis=None, vmin=-30, vmax=30)

pasar,Balongsari,Genteng Baru,Kembang,Pabean,Pucang Anom,Tambahrejo,Wonokromo
komoditas,,,,,,,
Bawang Bombay,6.000000,-2.700000,-1.200000,-3.900000,8.800000,-0.200000,-6.500000
Buncis,3.700000,-6.300000,-3.500000,-17.200000,12.000000,-5.000000,17.100000
Jagung Pipilan Kering (Jenis Kecil),12.400000,-11.000000,-0.200000,14.000000,-9.300000,-3.500000,-2.600000
Kacang Hijau,4.100000,-1.300000,-1.200000,-7.100000,-0.900000,1.000000,5.300000
Kacang Tanah (Jenis Besar),2.400000,0.800000,1.600000,-6.300000,-2.900000,1.900000,2.400000
Ketela Pohon,-8.300000,3.700000,-0.700000,-15.000000,27.300000,4.500000,-13.500000
Telur Ayam Kampung,8.400000,-1.300000,5.000000,-14.000000,-1.800000,6.900000,-3.200000
Tomat,-14.700000,6.500000,-1.800000,-16.400000,24.500000,1.300000,-0.400000
Udang Segar (Uk. Sedang),-4.200000,10.500000,1.400000,-4.800000,-3.100000,-1.000000,1.600000


In [12]:
pd.read_csv(ROOT / "data/staging/stg_hari_tanpa_data.csv").head()

,tanggal,alasan
0,2023-04-22,tidak ada tab
1,2023-04-23,tidak ada tab
2,2023-06-29,tidak ada tab
3,2025-03-31,tidak ada tab
4,2026-04-28,tab kosong
